In [1]:
#Make sure R can find packages installed with Conda
.libPaths('/hb/home/jbos/.conda/envs/vcfR')
.libPaths("/hb/home/jbos/.conda/envs/vcfR/lib/R/library")

In [2]:
#Load packages
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
leading_zeros<-function(num){
    if (nchar(as.character(num))<2){
        return(paste0('00',num))
        } else {
        if (nchar(as.character(num))<3){
            return(paste0('0',num))
            } else {
       return(as.character(num))
            }
        }
    }

In [4]:
angsdpca<-as.matrix(read.table('/scratch/jbos/Moz_aligned_mil/angsd_output137/angsd_acropora_pca2.cov'))

In [5]:
bamlist<-read.table('/home/jbos/Moz_reads/bam_names_grp137.txt')

In [6]:
bamlist_B<-read.table('/home/jbos/Moz_reads/bam_names_grp1.txt')

In [7]:
bamnames<-function(bam){
    a<-strsplit(bam,'files/')[[1]][2]
    b<-strsplit(a,'.s')[[1]][1]
    return(b)
    }

In [8]:
indlist<-apply(bamlist,FUN=bamnames,MARGIN=1)

In [9]:
indlist_B<-apply(bamlist_B,FUN=bamnames,MARGIN=1)

In [12]:
bleaching_all<-read.csv('/home/jbos/Moz_reads/Bleaching_Jaelyn.csv',header=TRUE)

In [11]:
other_metadat<-read.csv('/home/jbos/Moz_reads/Acropora_moz_metadat_certainty.csv')

In [13]:
colnames(bleaching_all)

[1] "Numero_do_tubo"      "Latitude"            "Longitude"          
[4] "Profundidade_metros" "Dia"                 "Mes"                
[7] "Ano"                 "Bleaching"           "Death"

In [14]:
#Add leading zero where necessary
bleaching_all$Numero_do_tubo<-sapply(bleaching_all$Numero_do_tubo,leading_zeros)
other_metadat$Numero_do_tubo<-sapply(other_metadat$Numero_do_tubo,leading_zeros)

In [15]:
bleaching_all$IND<-paste("ACR_",bleaching_all$Numero_do_tubo,sep="")
other_metadat$IND<-paste("ACR_",other_metadat$Numero_do_tubo,sep="")

In [16]:
other_metadat<-other_metadat[other_metadat$IND %in% indlist,]
bleaching_all<-bleaching_all[bleaching_all$IND %in% indlist,]

In [17]:
other_metadat<-other_metadat[,c('IND','Loc')]

In [18]:
metadat<-left_join(bleaching_all,other_metadat)

Joining with `by = join_by(IND)`


In [21]:
bleaching<-metadat[metadat$Ano=='2024',]
bleaching<-bleaching[bleaching$Latitude>(-14),]

In [23]:
bleaching$spp<-'A'
bleaching$spp[bleaching$IND %in% indlist_B]<-'B'

In [24]:
table(bleaching$spp)


 A  B 
76 60 

In [25]:
colnames(bleaching)

[1] "Numero_do_tubo"      "Latitude"            "Longitude"          
 [4] "Profundidade_metros" "Dia"                 "Mes"                
 [7] "Ano"                 "Bleaching"           "Death"              
[10] "IND"                 "Loc"                 "spp"

In [26]:
bay<-bleaching[bleaching$Loc=='V',]

In [27]:
colnames(bleaching)

[1] "Numero_do_tubo"      "Latitude"            "Longitude"          
 [4] "Profundidade_metros" "Dia"                 "Mes"                
 [7] "Ano"                 "Bleaching"           "Death"              
[10] "IND"                 "Loc"                 "spp"

In [28]:
table(bleaching$Bleach)


  0   1 
102  31 

In [29]:
table(bay$Bleaching)


 0  1 
45 28 

In [31]:
bay_B<-bay[bay$spp=='B',]
bay_A<-bay[bay$spp=='A',]

In [34]:
table(bay_B$Bleaching)
24/(24+35)


 0  1 
35 24 

[1] 0.4067797

In [35]:
table(bay_A$Bleaching)
4/14


 0  1 
10  4 

[1] 0.2857143

In [41]:
table(bay_A$Death)
table(bay_B$Death)


 0 
14 


 0  1 
48 11 